Bridge placement and classification - PV Capela
=
This notebook continues `Bridges - Capela.ipynb`. It starts from the Mesas drawing where the 16/32-module structure blocks are already inserted, places every bridge (the joint between two adjacent structures in a continuous row), classifies it by the structures it joins, and checks whether it falls inside a string (separates it) or at a string end.

**Workflow overview:**
1. Convert both DWGs to DXF and extract all model space entities.
2. Read the `EMF379_estrutura_16` / `_32` blocks, give each structure its X extent, and group them into continuous rows.
3. Read the string blocks and classify them as top/bottom (two-high table) or single, with their X extent.
4. For each pair of neighbouring structures in a row, create a bridge: type, centre X, row and `PRK`.
5. Mark the bridges that sit at a string end (within 2 m of a string start/end) as not affecting a string.
6. Export the bridges to Excel and write an AutoCAD script that draws a circle at each bridge, on two layers (affects a string / doesn't).

**Warning - written for an older version of `DWG_to_DuckDB_ETL.py`:**
- It calls `extract_data_from_dwf`, which the current module doesn't define (it only has `extract_data_from_dxf`), so the first cell fails with a `NameError`.
- It reads the block insertion point from `start_x` / `start_y`. The current module stores it in `X` / `Y`, so the block positions would come out empty (NaN) even after renaming the function.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **Files:** `PV_Capela - Mesas 16-32 - com estruturas.dwg` and `PV Capela - Eletromecânico + DC.dwg` in `C:\Users\Usuario\Desktop\projetos\capela`. Both are copied to `temp_folder` and converted together, so `temp_folder` must not hold any other DWG. The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** blocks inside other blocks, xrefs or paper space aren't read.
- **Structures drawing** (`... - com estruturas.dwg`):
  - One block reference per table, named exactly `EMF379_estrutura_16` or `EMF379_estrutura_32`. The module count is read from the **last 2 characters of the block name**.
  - Insertion point at the centre of the table, rotation 0 (the X extents are computed as centre ± half length, horizontally).
  - Every structure block has a `PRK` attribute (`1` / `2`). The insert script of `Bridges - Capela.ipynb` doesn't fill it, so it has to be set separately.
  - All the structures of one row have the same insertion Y (rounded to the metre).
  - The drawing must also contain at least one LINE/LWPOLYLINE, because the code drops `end_x`, `end_y`, `length` by name.
- **Electromechanical drawing** (`PV Capela - Eletromecânico + DC.dwg`):
  - One block reference per string, named exactly `EMF379_string_1` or `EMF379_string_2` (exact, case-sensitive), in model space, with the insertion point at the centre of the string.
  - Attributes `PRK`, `UFV`, `PST`, `INV`, `STR` are used to build the string tag. `PRK` must be `1` (Capela I) or `2` (Capela II).
  - The drawing must also contain entities that produce the columns `QBT`, `DSJ`, `POT`, `GRP`, `EFX` (attributes of some block) and `end_x`, `end_y`, `length` (at least one LINE/LWPOLYLINE), because the code drops them by name without `errors='ignore'`.
  - Two strings on the same table (two-high) have the same X (within 0.5 m) and are less than 5 m apart in Y. The table row centreline is 1.1613 m below the top string and 1.1613 m above the bottom string. A single string (type `C`) sits on the centreline.
  - The row Y of the strings (rounded to the metre) must be the same integer as the row Y of the structures, since the rows are matched with `==`.
- **Running `bridges.scr` in AutoCAD:** the layers `EMF_mec_bridges` and `EMF_mec_bridges_strings` must already exist in the target drawing (`LAYER S` only makes an existing layer current). Each bridge is drawn as a `CIRCLE` of radius 2.

Extracting the AutoCAD data from both drawings
=
- Copies the Mesas drawing (with the structure blocks) and the electromechanical drawing into `temp_folder` and converts both to DXF (ACAD2018) into `output_folder` with `convert_dwg_to_dxf`.
- `extract_data_from_dwf` reads each DXF into a list of dicts, one per model space entity. Nothing is loaded into DuckDB.
  - `data_STRUCT`: the structures drawing.
  - `data_ELM`: the electromechanical drawing (string blocks).
- Deletes the two DXFs and the two copied DWGs at the end.
- **Warning:** `extract_data_from_dwf` isn't defined in the current `DWG_to_DuckDB_ETL.py` (only `extract_data_from_dxf`), so this cell fails as it is.

In [149]:
import pandas as pd

from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\capela"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\capela\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\capela\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file1 = os.path.join(dwg_folder, "PV_Capela - Mesas 16-32 - com estruturas.dwg")
dwg_file2 = os.path.join(dwg_folder, "PV Capela - Eletromecânico + DC.dwg")

shutil.copy(dwg_file1, temp_folder)
shutil.copy(dwg_file2, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file1 = os.path.join(output_folder, "PV_Capela - Mesas 16-32 - com estruturas.dxf")
dxf_file2 = os.path.join(output_folder, "PV Capela - Eletromecânico + DC.dxf")

temp_dwg_file1 = os.path.join(temp_folder, "PV_Capela - Mesas 16-32 - com estruturas.dwg")
temp_dwg_file2 = os.path.join(temp_folder, "PV Capela - Eletromecânico + DC.dwg")

data_STRUCT = extract_data_from_dwf(dxf_file1)
print("Data Loaded")
data_ELM = extract_data_from_dwf(dxf_file2)
print("ELM Data Loaded")

os.remove(dxf_file1)
os.remove(dxf_file2)
os.remove(temp_dwg_file1)
os.remove(temp_dwg_file2)

Data Loaded
ELM Data Loaded


Reading the structure blocks
=
- Keeps only the blocks named `EMF379_estrutura_32` / `EMF379_estrutura_16`.
- `No of modules` = last 2 characters of the block name as an integer (`16` or `32`).
- `Position_X` = insertion X (`start_x` in the old module). `Position_Y` = insertion Y rounded to the metre: the row key. The exact Y stays in `start_y`.
- Drops the line-only columns and `rotation`.
- Sort order: `Position_Y` descending, then `Position_X` ascending (row by row, west to east).

In [150]:
mechanical_structures = pd.DataFrame(data_STRUCT)

mask = (mechanical_structures['name'] == "EMF379_estrutura_32") | (mechanical_structures['name'] == "EMF379_estrutura_16")

mechanical_structures = mechanical_structures[mask].copy()

mechanical_structures['No of modules'] = mechanical_structures['name'].str[-2:].astype(int)
mechanical_structures.rename(columns={'start_x':'Position_X'}, inplace=True)
mechanical_structures.drop(columns=['end_x', 'end_y', 'length', 'rotation'], inplace=True)
mechanical_structures['Position_Y'] = round(mechanical_structures['start_y'])
mechanical_structures.sort_values(by=['Position_Y', 'Position_X'], ascending=[False, True], inplace=True)
print(mechanical_structures.describe())
print(mechanical_structures.info())
print(mechanical_structures)

         Position_X        start_y  No of modules     Position_Y
count   6840.000000    6840.000000    6840.000000    6840.000000
mean  -18656.978463  157474.220567      20.023392  157474.411550
std      456.245415     599.696649       6.942163     599.705938
min   -19714.321000  156354.122600      16.000000  156354.000000
25%   -19016.491425  156877.622600      16.000000  156878.000000
50%   -18663.131600  157630.122600      16.000000  157630.000000
75%   -18283.609225  157987.622600      32.000000  157988.000000
max   -17791.646300  158304.622600      32.000000  158305.000000
<class 'pandas.core.frame.DataFrame'>
Index: 6840 entries, 5739 to 6838
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   type           6840 non-null   object 
 1   layer          6840 non-null   object 
 2   handle         6840 non-null   object 
 3   Position_X     6840 non-null   float64
 4   start_y        6840 non-null   float64
 5 

Structure X extents
=
`start_x` / `end_x` = centre ± half the table length: 32 modules ± 10.574 m, 16 modules ± 5.302 m. (`Bridges - Capela.ipynb` uses ± 5.282 for the 16s.)

In [151]:
mechanical_structures['start_x'] = np.where(
    mechanical_structures['No of modules'] == 32,
    mechanical_structures['Position_X'] - 10.574,
    mechanical_structures['Position_X'] - 5.302
)

mechanical_structures['end_x'] = np.where(
    mechanical_structures['No of modules'] == 32,
    mechanical_structures['Position_X'] + 10.574,
    mechanical_structures['Position_X'] + 5.302
)


print(mechanical_structures.describe())
print(mechanical_structures.info())
print(mechanical_structures)

         Position_X        start_y  No of modules     Position_Y      start_x  \
count   6840.000000    6840.000000    6840.000000    6840.000000   6840.00000   
mean  -18656.978463  157474.220567      20.023392  157474.411550 -18663.60617   
std      456.245415     599.696649       6.942163     599.705938    456.18434   
min   -19714.321000  156354.122600      16.000000  156354.000000 -19719.62300   
25%   -19016.491425  156877.622600      16.000000  156878.000000 -19024.57990   
50%   -18663.131600  157630.122600      16.000000  157630.000000 -18670.41240   
75%   -18283.609225  157987.622600      32.000000  157988.000000 -18290.10100   
max   -17791.646300  158304.622600      32.000000  158305.000000 -17796.94830   

              end_x  
count   6840.000000  
mean  -18650.350755  
std      456.317948  
min   -19709.019000  
25%   -19008.584600  
50%   -18656.746700  
75%   -18276.970250  
max   -17786.344300  
<class 'pandas.core.frame.DataFrame'>
Index: 6840 entries, 5739 to 6838


Grouping the structures into continuous rows
=
Same rules as in `Bridges - Capela.ipynb`. A new group starts when the row changes (`condition1`) or when there is a gap between two consecutive structures:
- `condition2`: no 16 on either side and centre-to-centre distance > 25 m.
- `condition3`: exactly one of the two is a 16, and the distance from this centre to the previous structure's start > 20 m.
- `condition4`: both are 16s, and the start-to-start distance > 15 m.

**Warning:** `No of modules` holds integers, but the conditions compare it with the string `"16"`. `condition3` and `condition4` never fire, and `condition2` applies to every pair (a plain "centre gap > 25 m" rule).

In [152]:
#Different Lines
condition1 = mechanical_structures["Position_Y"] != mechanical_structures["Position_Y"].shift()

#Break in the Structure with no 16 modules on the edges
condition2 = ((abs(mechanical_structures["Position_X"] - mechanical_structures["Position_X"].shift()) > 25) &
              (mechanical_structures["No of modules"] != "16") &
              (mechanical_structures["No of modules"].shift() != "16"))

#Break in the Structure with one 16 modules on the edges
condition3 = ((abs(mechanical_structures["Position_X"] - mechanical_structures["start_x"].shift()) > 20) &
              ((mechanical_structures["No of modules"] == "16") != (mechanical_structures["No of modules"].shift() == "16")))

#Break in the Structure with two 16 modules on the edges
condition4 = ((abs(mechanical_structures["start_x"] - mechanical_structures["start_x"].shift()) > 15) &
              (mechanical_structures["No of modules"] == "16") &
              (mechanical_structures["No of modules"].shift() == "16"))

break_condition = condition1 | condition2 | condition3 | condition4


mechanical_structures['Group'] = break_condition.cumsum()
print(mechanical_structures)

        type           layer handle  Position_X      start_y  \
5739  INSERT  ZAGO-MODULO_16   5284 -18354.9830  158304.6226   
5740  INSERT  ZAGO-MODULO_16   5285 -18344.3090  158304.6226   
3     INSERT  ZAGO-MODULO_32   3C1C -18328.2980  158304.6226   
5741  INSERT  ZAGO-MODULO_16   5286 -18312.2870  158304.6226   
4     INSERT  ZAGO-MODULO_32   3C1D -18296.2760  158304.6226   
...      ...             ...    ...         ...          ...   
6834  INSERT  ZAGO-MODULO_16   56CB -19069.8179  156362.1226   
6835  INSERT  ZAGO-MODULO_16   56CC -19112.5139  156354.1226   
6836  INSERT  ZAGO-MODULO_16   56CD -19101.8399  156354.1226   
6837  INSERT  ZAGO-MODULO_16   56CE -19091.1659  156354.1226   
6838  INSERT  ZAGO-MODULO_16   56CF -19080.4919  156354.1226   

                     name PRK  No of modules  Position_Y     start_x  \
5739  EMF379_estrutura_16   2             16    158305.0 -18360.2850   
5740  EMF379_estrutura_16   2             16    158305.0 -18349.6110   
3     EMF379_es

Reading the string blocks
=
Builds a DataFrame from the electromechanical drawing, drops `type`, `handle` and `rotation`, and keeps only the string blocks (`EMF379_string_1` / `EMF379_string_2`, exact names).

In [153]:
data_ELM_df = pd.DataFrame(data_ELM)
data_ELM_df.drop(columns=["type", "handle", "rotation"], inplace=True)

mask = (data_ELM_df['name'] == "EMF379_string_1") | (data_ELM_df['name'] == "EMF379_string_2")

strings_df = data_ELM_df[mask].copy()
print(strings_df.describe())
print(strings_df.info())
print(strings_df)

            start_x        start_y  end_x  end_y  length
count   4280.000000    4280.000000    0.0    0.0     0.0
mean  -18652.351424  157475.770896    NaN    NaN     NaN
std      457.690980     594.782582    NaN    NaN     NaN
min   -19698.300000  156352.961300    NaN    NaN     NaN
25%   -19008.387100  156923.041950    NaN    NaN     NaN
50%   -18667.613500  157632.541950    NaN    NaN     NaN
75%   -18280.373350  157986.783900    NaN    NaN     NaN
max   -17796.983300  158305.783900    NaN    NaN     NaN
<class 'pandas.core.frame.DataFrame'>
Index: 4280 entries, 43 to 7072
Data columns (total 17 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   layer    4280 non-null   object 
 1   name     4280 non-null   object 
 2   start_x  4280 non-null   float64
 3   start_y  4280 non-null   float64
 4   PRK      4280 non-null   object 
 5   UFV      4280 non-null   object 
 6   PST      4280 non-null   object 
 7   INV      4280 non-null   object 
 8  

Classifying the strings
=
Same logic as in `Bridges - Capela.ipynb`:
- `T` / `B`: another string within ±0.5 m in X and ±5 m in Y (two-high table), top or bottom. `C`: single string.
- `Line` = row centreline Y rounded to the metre (`C`: `middle_y`, `T`: `middle_y - 1.1613`, `B`: `middle_y + 1.1613`).
- X extent: `C` = `middle_x ± 10.574`, `T`/`B` = `middle_x ± 21.148`.
- Builds the `PRK.UFV.PST.INV.STR` tag and drops the other attribute columns.

In [154]:
strings_df.rename(columns={"start_x": "middle_x", "start_y": "middle_y"}, inplace=True)

for idx, row in strings_df.iterrows():
    x = row['middle_x']
    y = row['middle_y']

    mask = (
        (strings_df['middle_y'] >= y - 5) &
        (strings_df['middle_y'] <= y + 5) &
        (strings_df['middle_x'] >= x - 0.5) &
        (strings_df['middle_x'] <= x + 0.5) &
        (strings_df.index != idx)
    )

    if mask.any():
        y_neighbors = strings_df.loc[mask, 'middle_y']

        if (y > y_neighbors).any():
            strings_df.loc[idx, 'Type'] = "T"
        else:
            strings_df.loc[idx, 'Type'] = "B"

    else:
        strings_df.loc[idx, 'Type'] = "C"


strings_df.loc[:,'Line'] = np.where(
    strings_df["Type"] == "C",
    round(strings_df["middle_y"]),
    np.where(
        strings_df["Type"] == "T",
        round(strings_df["middle_y"] - 1.1613),
        round(strings_df["middle_y"] + 1.1613)
    )
)

strings_df['start_x'] = np.where(
    strings_df["Type"] == "C",
    strings_df['middle_x'] - 10.574,
    strings_df['middle_x'] - 21.148
)

strings_df['end_x'] = np.where(
    strings_df["Type"] == "C",
    strings_df['middle_x'] + 10.574,
    strings_df['middle_x'] + 21.148
)


strings_df.sort_values(by=["Line", "middle_x"], ascending=[False,True], inplace=True, ignore_index=True)
strings_df["PRK.UFV.PST.INV.STR"] = strings_df["PRK"].astype(str) + "." + strings_df["UFV"].astype(str) + "." + strings_df["PST"].astype(str) + "." + strings_df["INV"].astype(str) + "." + strings_df["STR"].astype(str)
strings_df.drop(columns=['layer', 'name', 'UFV', 'PST', 'INV', 'QBT', 'DSJ', 'POT', 'GRP', 'EFX', 'STR', 'end_y', 'length'], inplace=True)
print(strings_df)

        middle_x     middle_y PRK       end_x Type      Line     start_x  \
0    -18349.6460  158304.6226   2 -18339.0720    C  158305.0 -18360.2200   
1    -18317.6140  158305.7839   2 -18296.4660    T  158305.0 -18338.7620   
2    -18317.6140  158303.4613   2 -18296.4660    B  158305.0 -18338.7620   
3    -18274.9180  158305.7839   2 -18253.7700    T  158305.0 -18296.0660   
4    -18274.9180  158303.4613   2 -18253.7700    B  158305.0 -18296.0660   
...          ...          ...  ..         ...  ...       ...         ...   
4275 -19107.1669  156363.2839   1 -19086.0189    T  156362.0 -19128.3149   
4276 -19107.1669  156360.9613   1 -19086.0189    B  156362.0 -19128.3149   
4277 -19075.1549  156362.1226   1 -19064.5809    C  156362.0 -19085.7289   
4278 -19096.4929  156355.2839   1 -19075.3449    T  156354.0 -19117.6409   
4279 -19096.4929  156352.9613   1 -19075.3449    B  156354.0 -19117.6409   

     PRK.UFV.PST.INV.STR  
0             2.1.2.1.01  
1             2.1.2.1.02  
2     

Placing the bridges
=
- Half gaps between neighbouring structures (m): 16-16 = 0.09/2, 32-16 = 0.175/2, 16-32 = 0.115/2, 32-32 = 0.2/2. The bridge centre = end of the left structure + half gap.
- For each row group (sorted by X), every pair of neighbours creates one bridge with: type, centre X, the exact Y of the left structure (`start_y`), the row key (`Line`), `PRK` (or `left_right` when the two structures belong to different plants) and `Affects_String = True`.
- **Warning:** the type labels are swapped. Two 32s get the label `'16-32'`, and a 16/32 mix gets `'32-32'`. As a result, the mixed pairs use the 32-32 gap, and the real 32-32 pairs fall through to the 32-16 gap. Only the 16-16 bridges are labelled and placed correctly.
- The table grows one row at a time with `bridges_df.loc[...]` (slow, but correct).

In [155]:
distance_between_16_16 = 0.09/2
distance_between_32_16 = 0.175/2
distance_between_16_32 = 0.115/2
distance_between_32_32 = 0.2/2

bridges_df = pd.DataFrame(columns=['Type', 'Position_X', 'Position_Y', 'Line', 'PRK', 'Affects_String'])
bridge_id = 0

for group_id, group_df in mechanical_structures.groupby('Group'):
    group_df = group_df.sort_values('Position_X').reset_index(drop=True)

    for i in range(len(group_df) - 1):
        struct_left = group_df.iloc[i]
        struct_right = group_df.iloc[i+1]


        left_type = struct_left['No of modules']
        right_type = struct_right['No of modules']

        left_PRK = struct_left['PRK']
        right_PRK = struct_right['PRK']

        if left_type == 16 and right_type == 16:
            bridge_type = '16-16'
        elif left_type == 32 and right_type == 32:
            bridge_type = '16-32'
        else:
            bridge_type = '32-32'


        if bridge_type == '16-16':
            bridge_center = struct_left['end_x'] + distance_between_16_16
        elif bridge_type == '32-32':
            bridge_center = struct_left['end_x'] + distance_between_32_32
        elif left_type == 16:
            bridge_center = struct_left['end_x'] + distance_between_16_32
        else:
            bridge_center = struct_left['end_x'] + distance_between_32_16


        if left_PRK == right_PRK:
            bridge_PRK = left_PRK
        else:
            bridge_PRK = str(left_PRK)+"_"+str(right_PRK)

        bridges_df.loc[bridge_id] = [bridge_type, bridge_center, struct_left['start_y'], struct_left['Position_Y'], bridge_PRK, True]
        bridge_id += 1

print(bridges_df)

       Type  Position_X   Position_Y      Line PRK  Affects_String
0     16-16 -18349.6360  158304.6226  158305.0   2            True
1     32-32 -18338.9070  158304.6226  158305.0   2            True
2     32-32 -18317.6240  158304.6226  158305.0   2            True
3     32-32 -18306.8850  158304.6226  158305.0   2            True
4     32-32 -18285.6020  158304.6226  158305.0   2            True
...     ...         ...          ...       ...  ..             ...
6147  16-16 -19085.8189  156362.1226  156362.0   1            True
6148  16-16 -19075.1449  156362.1226  156362.0   1            True
6149  16-16 -19107.1669  156354.1226  156354.0   1            True
6150  16-16 -19096.4929  156354.1226  156354.0   1            True
6151  16-16 -19085.8189  156354.1226  156354.0   1            True

[6152 rows x 6 columns]


Checking the bridge table
=
`describe()` / `info()` of the bridges (count, X/Y range, column types).

In [156]:
print(bridges_df.describe())
print(bridges_df.info())

         Position_X     Position_Y           Line
count   6152.000000    6152.000000    6152.000000
mean  -18651.916490  157480.388506  157480.578186
std      458.862866     596.045871     596.055054
min   -19708.974000  156354.122600  156354.000000
25%   -19017.680100  156909.622600  156910.000000
50%   -18649.115200  157638.122600  157638.000000
75%   -18281.897375  157987.622600  157988.000000
max   -17796.983300  158304.622600  158305.000000
<class 'pandas.core.frame.DataFrame'>
Index: 6152 entries, 0 to 6151
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Type            6152 non-null   object 
 1   Position_X      6152 non-null   float64
 2   Position_Y      6152 non-null   float64
 3   Line            6152 non-null   float64
 4   PRK             6152 non-null   object 
 5   Affects_String  6152 non-null   bool   
dtypes: bool(1), float64(3), object(2)
memory usage: 294.4+ KB
None


Bridges at a string end
=
- For each bridge, looks for a string on the same row (`Line`) whose start or end X is within ±2 m of the bridge centre.
- If there is one, the bridge sits at a string end and `Affects_String` becomes `False`. The remaining `True` bridges fall inside a string and separate it.
- Prints how many bridges are in each case.

In [157]:
for idx, row in bridges_df.iterrows():

    left_boundary = row['Position_X'] - 2
    right_boundary = row['Position_X'] + 2
    # Filter structures

    mask = (
        (strings_df['Line'] == row['Line']) &
        (
            (strings_df['start_x'].between(left_boundary, right_boundary)) |
            (strings_df['end_x'].between(left_boundary, right_boundary))
        )
    )

    if mask.any():
        bridges_df.loc[idx, 'Affects_String'] = False

print(bridges_df['Affects_String'].value_counts())

Affects_String
True     4852
False    1300
Name: count, dtype: int64


Exporting the bridges and the AutoCAD script
=
- `Group` = `PRK_Type_AffectsString` (e.g. `2_32-32_True`), counted with `value_counts()`. Since the type labels are swapped (see *Placing the bridges*), `16-32` / `32-32` here don't mean what they say.
- `Script` = `CIRCLE x,y 2`: a circle of radius 2 at the bridge centre.
- Exports the table to `C:\Users\Usuario\Desktop\teste.xlsx` (hardcoded path).
- Writes `C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Bridges\bridges.scr`:
  - `LAYER S EMF_mec_bridges`, then the circles of the bridges that don't affect a string.
  - `LAYER S EMF_mec_bridges_strings`, then the circles of the bridges that separate a string.
  - The empty line after each `LAYER S` ends the `LAYER` command. Both layers must already exist in the drawing.

In [163]:
bridges_df['Group'] = bridges_df.apply(lambda row: row['PRK'] + "_" + row['Type'] + "_" + str(row['Affects_String']), axis=1)
bridges_df["Script"] = bridges_df.apply(lambda row: (f'CIRCLE {round(row['Position_X'],4)},{round(row['Position_Y'],4)} 2'), axis=1)

print(bridges_df)
print(bridges_df['Group'].value_counts())
bridges_df.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Bridges\bridges.scr", 'w') as f:
    f.write("LAYER S EMF_mec_bridges\n\n")
    f.write('\n'.join(bridges_df[bridges_df['Affects_String'] == False]["Script"].astype(str)) + '\n')
    f.write("LAYER S EMF_mec_bridges_strings\n\n")
    f.write('\n'.join(bridges_df[bridges_df['Affects_String'] == True]["Script"].astype(str)) + '\n')


       Type  Position_X   Position_Y      Line PRK  Affects_String  \
0     16-16 -18349.6360  158304.6226  158305.0   2            True   
1     32-32 -18338.9070  158304.6226  158305.0   2           False   
2     32-32 -18317.6240  158304.6226  158305.0   2            True   
3     32-32 -18306.8850  158304.6226  158305.0   2            True   
4     32-32 -18285.6020  158304.6226  158305.0   2            True   
...     ...         ...          ...       ...  ..             ...   
6147  16-16 -19085.8189  156362.1226  156362.0   1           False   
6148  16-16 -19075.1449  156362.1226  156362.0   1            True   
6149  16-16 -19107.1669  156354.1226  156354.0   1            True   
6150  16-16 -19096.4929  156354.1226  156354.0   1            True   
6151  16-16 -19085.8189  156354.1226  156354.0   1            True   

              Group                            Script  
0      2_16-16_True   CIRCLE -18349.636,158304.6226 2  
1     2_32-32_False   CIRCLE -18338.907,158304.